In [1]:
!pip install findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Funciones drop, sample, randomSplit Parte 5 ").getOrCreate()
sc = spark.sparkContext

In [2]:
df = spark.read.parquet("./data")

## Funcion drop

# Eliminación de Columnas: `drop()`

La transformación `drop()` se utiliza simplemente para eliminar una o más columnas específicas de un DataFrame.

### Puntos clave a tener en cuenta:

* **Flexibilidad:** Permite especificar uno o varios nombres de columnas simultáneamente para ser removidas.
* **Tolerancia a errores:** Solo se eliminarán las columnas que existan actualmente dentro del esquema (*Schema*). Si indicas el nombre de una columna que no existe, Spark la **ignorará automáticamente** sin lanzar ningún tipo de error o excepción.

---

## Ejemplos de uso en PySpark

### 1. Eliminar una sola columna
```python
# Elimina únicamente la columna llamada "edad"
df_limpio = df.drop("edad")

In [4]:
df.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: integer (nullable = true)
 |-- likes: integer (nullable = true)
 |-- dislikes: integer (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



In [5]:
df_util = df.drop("comments_disabled")

In [6]:
df_util.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: integer (nullable = true)
 |-- likes: integer (nullable = true)
 |-- dislikes: integer (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



In [7]:
df_util = df.drop("comments_disabled","likes","dislikes","video_error_or_removed")

In [8]:
df_util.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: integer (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- description: string (nullable = true)



In [10]:
# ATENCION drop no nos va arrojar un error si no se encuentra en el Schema

df_util = df_util.drop("vida")

# La Transformación `sample()`

`sample()` es una transformación que devuelve un subconjunto de filas seleccionadas de forma aleatoria a partir del DataFrame original.

### Características Principales:

* **Tamaño de la muestra (Fracción):** El número de filas devueltas es **aproximadamente igual** al porcentaje especificado. Este valor se define como una fracción que debe oscilar estrictamente entre **0.0 y 1.0** (por ejemplo, `0.1` representa el 10% de los datos y `0.5` el 50%).
* **Resultado aproximado:** Debido a la naturaleza distribuida de Spark (donde el muestreo se realiza de forma independiente en cada partición), el número final de filas obtenidas es una aproximación estadística y puede variar ligeramente entre ejecuciones.

---

## Sintaxis y Parámetros en PySpark

El método `sample()` cuenta con tres parámetros clave para controlar el muestreo:

```python
df_muestreado = df.sample(withReplacement=False, fraction=0.1, seed=42)

In [13]:
# sample (muestreo) 80% en este caso

df_muestra = df.sample(0.8)

In [17]:
num_de_filar = df_muestra.count()

num_de_filas_original = df.count()

In [18]:
print(num_de_filar)

38449


In [19]:
print(num_de_filas_original)

48137


In [22]:
# lo que vamos a generarle com el parametro seed es la semilla para que nos de esa aleatoriedad

df_muestra = df.sample(fraction=0.8, seed=1234)

In [23]:
# este parametro hace un remplazamiento de filas por default viene en FALSE

df_muestra = df.sample(withReplacement=True, fraction=0.8, seed=1234)

# La Transformación `randomSplit()`

`randomSplit()` es una herramienta fundamental utilizada comúnmente durante la fase de preparación de datos en proyectos de **Machine Learning**.

A diferencia de la mayoría de las transformaciones tradicionales de Spark, esta operación tiene la particularidad de **devolver una lista o arreglo con múltiples DataFrames independientes**.

---

## ¿Cómo funciona?

La cantidad de DataFrames devueltos y la proporción de filas que recibirá cada uno se basa estrictamente en el arreglo de **pesos (*weights*)** que especifiquemos como parámetro.



### Puntos clave a tener en cuenta:
* **Normalización secuencial:** Si el conjunto de pesos introducido no suma exactamente 1 (por ejemplo, `[3, 1]`), debemos tener en cuenta que Spark **normalizará estos pesos en secuencia** de forma automática para que su suma sea igual a 1 (transformando el ejemplo interno en `[0.75, 0.25]`). Aun así, se recomienda que sumen 1.0 por buenas prácticas y legibilidad del código.
* **Resultados aproximados:** Al igual que con `sample()`, la división de registros se calcula de forma aleatoria e independiente en cada partición, por lo que el tamaño final de los DataFrames resultantes será estadísticamente aproximado a los pesos indicados.

---

## Ejemplo de uso en PySpark (División Train/Test)

El caso de uso más habitual de `randomSplit()` es separar un conjunto de datos en un grupo de **entrenamiento (Train)** y otro de **prueba (Test)**:

```python
# Dividimos el DataFrame original en 80% para entrenar y 20% para evaluar
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# Ahora podemos operar con ambos DataFrames por separado
print(f"Registros para entrenamiento: {train_df.count()}")
print(f"Registros para prueba: {test_df.count()}")

In [24]:
# randomSplit (empaquetamos las variables) DEBEN SUMAR 1

train, test = df.randomSplit([0.8, 0.2], seed=1234)

In [28]:
train.count()

test.count()

print(f"Registros para entrenamiento: {train.count()}")
print(f"Registros para prueba: {test.count()}")

Registros para entrenamiento: 38506
Registros para prueba: 9631
